## Install required data

[Amazon aws](https://s3.amazonaws.com/tripdata/index.html)

put into data folder
data/2021-citibike-tripdata


In [1]:
import pandas as pd
import numpy as np

In [22]:
from pathlib import Path
import pandas as pd
import numpy as np

def process_citibike_data():
    # Read all CSVs
    all_dfs = []
    for csv_file in Path("../data").glob("**/*.csv"):
        df = pd.read_csv(csv_file)
        all_dfs.append(df)
    
    df = pd.concat(all_dfs, ignore_index=True)
    
    # Convert datetime columns
    df['started_at'] = pd.to_datetime(df['started_at'])
    df['ended_at'] = pd.to_datetime(df['ended_at'])
    
    # Clean data
    df = df.dropna(subset=['started_at', 'ended_at'])  # Remove trips with missing times
    
    # Feature engineering
    df['duration_minutes'] = (df['ended_at'] - df['started_at']).dt.total_seconds() / 60
    df['hour'] = df['started_at'].dt.hour
    df['day_of_week'] = df['started_at'].dt.day_name()
    df['month'] = df['started_at'].dt.month
    
    
    return df

df = process_citibike_data()
print(df.head())

/var/folders/fy/rf5kskyx2x3_l2ff4vm40vrh0000gn/T/ipykernel_92409/3385014502.py:9: DtypeWarning: Columns (7: end_station_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_file)
/var/folders/fy/rf5kskyx2x3_l2ff4vm40vrh0000gn/T/ipykernel_92409/3385014502.py:9: DtypeWarning: Columns (5: start_station_id, 7: end_station_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_file)


### Extract needed columns

In [27]:
cleaned_df = df[['ride_id', 'started_at', 'ended_at', 'start_station_name', 'end_station_name', 'day_of_week']]

## Group by Hour
other options:
- daily, weekly, etc..
- start_station_name/end_station_name
- 


In [28]:
hourly = cleaned_df.groupby(pd.Grouper(key='started_at', freq='h')).agg({
    'ride_id': 'count',
}).rename(columns={'ride_id': 'trip_count'})

In [25]:
hourly

,trip_count,duration_minutes,distance_miles
started_at,,,
2021-01-30 17:00:00,1,687646.693117,1.622619
2021-01-30 18:00:00,0,NaN,NaN
2021-01-30 19:00:00,0,NaN,NaN
2021-01-30 20:00:00,0,NaN,NaN
2021-01-30 21:00:00,0,NaN,NaN
...,...,...,...
2022-12-31 19:00:00,1104,12.339558,0.995206
2022-12-31 20:00:00,899,13.382851,1.112379
2022-12-31 21:00:00,517,12.545554,1.061769


### Group by station

In [30]:
# Group by start station
station_stats = df.groupby('start_station_name').agg({
    'ride_id': 'count',
    'member_casual': lambda x: (x == 'member').sum()  # count members
}).rename(columns={
    'ride_id': 'total_trips',
    'member_casual': 'member_count'
})

station_stats['casual_count'] = station_stats['total_trips'] - station_stats['member_count']
station_stats = station_stats.sort_values('total_trips', ascending=False)

print(station_stats.head(10))

                         total_trips  member_count  casual_count
start_station_name                                              
W 21 St & 6 Ave               129018        112057         16961
West St & Chambers St         123291         83566         39725
Broadway & W 58 St            114293         83658         30635
6 Ave & W 33 St               106445         83813         22632
1 Ave & E 68 St               104856         93711         11145
Broadway & E 14 St             98866         80361         18505
Broadway & W 25 St             98425         76083         22342
University Pl & E 14 St        97126         80884         16242
Broadway & E 21 St             95691         77411         18280
W 31 St & 7 Ave                94208         76318         17890
